# Sequence Training Guide

This notebook provides a comprehensive guide to training models in the Sequence framework.

## Contents
1. [Environment Setup](#setup)
2. [Data Preparation](#data-prep)
3. [Supervised Training](#supervised)
4. [Multi-Task Training](#multitask)
5. [Reinforcement Learning](#rl)
6. [Evaluation](#evaluation)

---

## 1. Environment Setup <a id="setup"></a>

First, let's set up the Python path and import necessary modules.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "run") not in sys.path:
    sys.path.insert(0, str(ROOT / "run"))

print(f"Project root: {ROOT}")
print(f"Python version: {sys.version}")

In [ ]:
# Core imports
import os
import torch
import pandas as pd
import numpy as np
from datetime import datetime

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Data Preparation <a id="data-prep"></a>

### Option A: Using Existing Prepared Data

If you already have prepared data in `data/data/`, you can skip to the training section.

In [ ]:
# Check for existing prepared data
data_dir = ROOT / "data" / "data"
if data_dir.exists():
    pairs = [p.name for p in data_dir.iterdir() if p.is_dir()]
    print(f"Found {len(pairs)} prepared pairs: {', '.join(pairs)}")
    
    # Check one pair in detail
    if pairs:
        sample_pair = pairs[0]
        prepared_file = data_dir / sample_pair / f"{sample_pair}_prepared.csv"
        if prepared_file.exists():
            df = pd.read_csv(prepared_file, nrows=5)
            print(f"\nSample from {sample_pair}:")
            print(f"Columns: {list(df.columns)}")
            print(f"Shape (first 5 rows): {df.shape}")
else:
    print("No prepared data found. Run data preparation first.")

### Option B: Prepare New Data

Run this cell to prepare data for a specific pair. Make sure you have raw HistData CSVs in `output_central/`.

In [ ]:
# Configuration for data preparation
PAIR = "gbpusd"  # Change to your desired pair
T_IN = 120       # Input sequence length
T_OUT = 10       # Output prediction horizon
USE_INTRINSIC_TIME = True
DC_THRESHOLD = 0.0005  # Directional-change threshold (5 pips for FX)
INCLUDE_SENTIMENT = False  # Set to True to include GDELT sentiment

print(f"Configuration:")
print(f"  Pair: {PAIR}")
print(f"  Input length: {T_IN}")
print(f"  Output horizon: {T_OUT}")
print(f"  Intrinsic time: {USE_INTRINSIC_TIME}")
print(f"  DC threshold: {DC_THRESHOLD}")
print(f"  Sentiment: {INCLUDE_SENTIMENT}")

In [ ]:
# Run data preparation (uncomment to execute)
# !python {ROOT}/data/prepare_dataset.py \
#     --pairs {PAIR} \
#     --t-in {T_IN} \
#     --t-out {T_OUT} \
#     --task-type classification \
#     {'--intrinsic-time' if USE_INTRINSIC_TIME else ''} \
#     {'--dc-threshold-up ' + str(DC_THRESHOLD) if USE_INTRINSIC_TIME else ''} \
#     {'--include-sentiment' if INCLUDE_SENTIMENT else ''}

## 3. Supervised Training <a id="supervised"></a>

Train a CNN-LSTM-Attention hybrid model for price prediction.

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'pair': 'gbpusd',
    'epochs': 50,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'checkpoint_dir': str(ROOT / 'models'),
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

### Method 1: Using the Training Script (Recommended)

This uses the tested `run_training.py` script.

In [ ]:
# Run supervised training
!python {ROOT}/train/run_training.py \
    --pairs {TRAINING_CONFIG['pair']} \
    --epochs {TRAINING_CONFIG['epochs']} \
    --learning-rate {TRAINING_CONFIG['learning_rate']} \
    --batch-size {TRAINING_CONFIG['batch_size']} \
    --device {TRAINING_CONFIG['device']}

### Method 2: Programmatic Training

For more control, you can train directly using the core training modules.

In [ ]:
# Import training modules
from config.config import DataConfig, ModelConfig, TrainingConfig
from data.iterable_dataset import create_dataloaders
from models.agent_hybrid import SharedEncoder
from train.training_manager import TrainingManager

# Create configurations
data_cfg = DataConfig(
    pair=TRAINING_CONFIG['pair'],
    t_in=120,
    t_out=10,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
)

model_cfg = ModelConfig(
    lstm_hidden_size=128,
    cnn_filters=[32, 64, 128],
    attention_dim=64,
    num_attention_heads=4,
    use_optimized_attention=False,  # Set True for sequences > 1024
)

train_cfg = TrainingConfig(
    epochs=TRAINING_CONFIG['epochs'],
    batch_size=TRAINING_CONFIG['batch_size'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    device=TRAINING_CONFIG['device'],
    checkpoint_dir=TRAINING_CONFIG['checkpoint_dir'],
)

print("Configurations created successfully!")

In [ ]:
# Create dataloaders
data_path = ROOT / "data" / "data" / data_cfg.pair / f"{data_cfg.pair}_prepared.csv"
print(f"Loading data from: {data_path}")

train_loader, val_loader, test_loader = create_dataloaders(
    data_path=str(data_path),
    batch_size=train_cfg.batch_size,
    t_in=data_cfg.t_in,
    t_out=data_cfg.t_out,
    train_ratio=data_cfg.train_ratio,
    val_ratio=data_cfg.val_ratio,
    num_workers=4,
)

print(f"Dataloaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Create model
# First, get input dimension from a sample batch
sample_batch = next(iter(train_loader))
input_dim = sample_batch[0].shape[-1]  # (batch, seq_len, features)
print(f"Input dimension: {input_dim}")

model = SharedEncoder(
    input_dim=input_dim,
    lstm_hidden_size=model_cfg.lstm_hidden_size,
    cnn_filters=model_cfg.cnn_filters,
    attention_dim=model_cfg.attention_dim,
    num_attention_heads=model_cfg.num_attention_heads,
    num_classes=3,  # Classification: down, neutral, up
)

model = model.to(train_cfg.device)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Initialize training manager and train
manager = TrainingManager(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
)

# Start training (uncomment to run)
# history = manager.train()
# print("Training complete!")

## 4. Multi-Task Training <a id="multitask"></a>

Train a model to jointly predict price movement, volatility, and market regime.

In [ ]:
# Multi-task training configuration
MULTITASK_CONFIG = {
    'pair': 'gbpusd',
    'epochs': 50,
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print("Multi-Task Configuration:")
for key, value in MULTITASK_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Run multi-task training
!python {ROOT}/train/run_training_multitask.py \
    --pairs {MULTITASK_CONFIG['pair']} \
    --epochs {MULTITASK_CONFIG['epochs']} \
    --batch-size {MULTITASK_CONFIG['batch_size']} \
    --device {MULTITASK_CONFIG['device']}

## 5. Reinforcement Learning <a id="rl"></a>

Train an A3C agent for optimal execution policy.

**Note**: RL training requires a prepared signal model or uses the execution environment directly.

In [ ]:
# RL training configuration
RL_CONFIG = {
    'pair': 'gbpusd',
    'env_mode': 'backtesting',  # 'backtesting' or 'simulated'
    'num_workers': 8,
    'total_steps': 1_000_000,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print("RL Configuration:")
for key, value in RL_CONFIG.items():
    print(f"  {key}: {value}")

print("\nEnvironment Modes:")
print("  - backtesting: Deterministic historical replay (reproducible)")
print("  - simulated: Stochastic retail execution with spread/slippage")

In [ ]:
# Prepare historical data path for RL
historical_data = ROOT / "data" / "data" / RL_CONFIG['pair'] / f"{RL_CONFIG['pair']}_prepared.csv"
print(f"Historical data: {historical_data}")
print(f"Exists: {historical_data.exists()}")

if historical_data.exists():
    df = pd.read_csv(historical_data, nrows=5)
    print(f"Data shape (preview): {df.shape}")
    print(f"Columns: {list(df.columns)[:10]}...")

In [ ]:
# Run RL training
!python {ROOT}/rl/run_a3c_training.py \
    --pair {RL_CONFIG['pair']} \
    --env-mode {RL_CONFIG['env_mode']} \
    --historical-data {historical_data} \
    --num-workers {RL_CONFIG['num_workers']} \
    --total-steps {RL_CONFIG['total_steps']} \
    --device {RL_CONFIG['device']}

## 6. Evaluation <a id="evaluation"></a>

Evaluate trained models on the test set.

In [ ]:
# Find available checkpoints
models_dir = ROOT / "models"
if models_dir.exists():
    checkpoints = list(models_dir.glob("*.pt")) + list(models_dir.glob("**/best_model.pt"))
    print(f"Found {len(checkpoints)} checkpoints:")
    for ckpt in checkpoints[:5]:  # Show first 5
        print(f"  {ckpt.relative_to(ROOT)}")
else:
    print("No models directory found. Train a model first.")

In [ ]:
# Evaluation configuration
EVAL_CONFIG = {
    'pair': 'gbpusd',
    'checkpoint_path': 'models/gbpusd_best_model.pt',  # Update with your checkpoint
}

print("Evaluation Configuration:")
for key, value in EVAL_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Run evaluation
checkpoint_path = ROOT / EVAL_CONFIG['checkpoint_path']
if checkpoint_path.exists():
    !python {ROOT}/eval/run_evaluation.py \
        --pairs {EVAL_CONFIG['pair']} \
        --checkpoint-path {checkpoint_path}
else:
    print(f"Checkpoint not found: {checkpoint_path}")
    print("Train a model first or update the checkpoint path.")

### Ensemble Evaluation with TimesFM

Benchmark against Google's TimesFM foundation model.

In [ ]:
# TimesFM ensemble evaluation
ENSEMBLE_CONFIG = {
    'pair': 'gbpusd',
    'years': [2023],
    't_in': 120,
    't_out': 10,
    'checkpoint_root': 'models',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print("Ensemble Configuration:")
for key, value in ENSEMBLE_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Run ensemble evaluation (uncomment to execute)
# !python {ROOT}/eval/ensemble_timesfm.py \
#     --pairs {ENSEMBLE_CONFIG['pair']} \
#     --years {' '.join(map(str, ENSEMBLE_CONFIG['years']))} \
#     --t-in {ENSEMBLE_CONFIG['t_in']} \
#     --t-out {ENSEMBLE_CONFIG['t_out']} \
#     --checkpoint-root {ENSEMBLE_CONFIG['checkpoint_root']} \
#     --device {ENSEMBLE_CONFIG['device']}

## 7. Unified Training Pipeline

For a complete end-to-end workflow, use the unified pipeline script.

In [ ]:
# Unified pipeline configuration
PIPELINE_CONFIG = {
    'pairs': 'gbpusd',
    'run_histdata_download': False,  # Set True if you need to download data
    'epochs': 50,
    'run_rl_training': True,
    'rl_env_mode': 'backtesting',
    'rl_num_workers': 8,
}

print("Pipeline Configuration:")
for key, value in PIPELINE_CONFIG.items():
    print(f"  {key}: {value}")

print("\nThis will run: Download → Prepare → Train → RL Training")

In [ ]:
# Run unified pipeline (uncomment to execute)
# !python {ROOT}/run/training_pipeline.py \
#     --pairs {PIPELINE_CONFIG['pairs']} \
#     {'--run-histdata-download' if PIPELINE_CONFIG['run_histdata_download'] else ''} \
#     --epochs {PIPELINE_CONFIG['epochs']} \
#     {'--run-rl-training' if PIPELINE_CONFIG['run_rl_training'] else ''} \
#     --rl-env-mode {PIPELINE_CONFIG['rl_env_mode']} \
#     --rl-num-workers {PIPELINE_CONFIG['rl_num_workers']}

## 8. Quick Test Commands

Useful commands for testing and validation.

In [ ]:
# Run all tests
!pytest {ROOT}/tests/ -v

In [ ]:
# Run only fast tests
!pytest {ROOT}/tests/ -m fast -v

In [ ]:
# Validate prepared data
!python {ROOT}/run/scripts/validate_training_data.py \
    --data-path {ROOT}/data/data/gbpusd/gbpusd_prepared.csv

In [ ]:
# Check code quality
!ruff check {ROOT}/train/ {ROOT}/models/ --select E,F,W

## 9. Monitoring and Debugging

### Check GPU Memory

In [ ]:
if torch.cuda.is_available():
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"GPU Memory Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
else:
    print("No GPU available")

### View Training Logs

In [ ]:
# Check for log files
log_files = list(ROOT.glob("*.log")) + list(ROOT.glob("logs/*.log"))
if log_files:
    print(f"Found {len(log_files)} log files:")
    for log in log_files:
        print(f"  {log.relative_to(ROOT)}")
else:
    print("No log files found.")

---

## Next Steps

1. **Data Preparation**: Ensure you have prepared data before training
2. **Start Small**: Begin with a single pair and short training (10 epochs) to verify setup
3. **Monitor GPU**: Watch GPU memory usage to avoid OOM errors
4. **Checkpointing**: Models are saved to `models/` directory automatically
5. **Testing**: Run tests frequently to ensure code correctness

For more details, see:
- [CLAUDE.md](../CLAUDE.md) - Complete architecture and development guide
- [docs/](../docs/) - Detailed documentation
- [tests/](../tests/) - Example test files

---

**Created**: 2026-01-12  
**Framework**: Sequence Deep Learning for FX Trading